## Bronze Layer – Raw Data Ingestion  
**Project:** Olist E-Commerce Data Analytics  

This notebook ingests raw Olist CSV files into Databricks Delta Lake Bronze tables using PySpark.  
The Bronze layer stores raw data from the source with minimal processing, allowing easy tracking and reprocessing of the data.

- **Source:** Olist Brazilian E-Commerce Dataset  
- **Storage:** Databricks Delta Lake (Bronze tables)  
- **Ingestion Method:** PySpark CSV ingestion from Unity Catalog Volume  
- **Transformations:** None (raw data preserved with applied schema)


In [0]:
%fs ls /Volumes/workspace/olist_ecom/olist_ecom_rawdata

path,name,size,modificationTime
dbfs:/Volumes/workspace/olist_ecom/olist_ecom_rawdata/olist_customers_dataset.csv,olist_customers_dataset.csv,9033957,1769488941000
dbfs:/Volumes/workspace/olist_ecom/olist_ecom_rawdata/olist_geolocation_dataset.csv,olist_geolocation_dataset.csv,61273883,1769488943000
dbfs:/Volumes/workspace/olist_ecom/olist_ecom_rawdata/olist_order_items_dataset.csv,olist_order_items_dataset.csv,15438671,1769488941000
dbfs:/Volumes/workspace/olist_ecom/olist_ecom_rawdata/olist_order_payments_dataset.csv,olist_order_payments_dataset.csv,5777138,1769488940000
dbfs:/Volumes/workspace/olist_ecom/olist_ecom_rawdata/olist_order_reviews_dataset.csv,olist_order_reviews_dataset.csv,14451670,1769488941000
dbfs:/Volumes/workspace/olist_ecom/olist_ecom_rawdata/olist_orders_dataset.csv,olist_orders_dataset.csv,17654914,1769488942000
dbfs:/Volumes/workspace/olist_ecom/olist_ecom_rawdata/olist_products_dataset.csv,olist_products_dataset.csv,2379446,1769488940000
dbfs:/Volumes/workspace/olist_ecom/olist_ecom_rawdata/olist_sellers_dataset.csv,olist_sellers_dataset.csv,174703,1769488940000
dbfs:/Volumes/workspace/olist_ecom/olist_ecom_rawdata/product_category_name_translation.csv,product_category_name_translation.csv,2613,1769488940000


In [0]:
%sql CREATE DATABASE IF NOT EXISTS olist_bronze;


In [0]:
# load the raw data 
cust_df = spark.read.csv('/Volumes/workspace/olist_ecom/olist_ecom_rawdata/olist_customers_dataset.csv',header=True,inferSchema=True)
orders_df = spark.read.csv('/Volumes/workspace/olist_ecom/olist_ecom_rawdata/olist_orders_dataset.csv',header=True,inferSchema=True)

geoloc_df = spark.read.csv('/Volumes/workspace/olist_ecom/olist_ecom_rawdata/olist_geolocation_dataset.csv',header=True,inferSchema=True)
orditems_df = spark.read.csv('/Volumes/workspace/olist_ecom/olist_ecom_rawdata/olist_order_items_dataset.csv',header=True,inferSchema=True)
payments_df = spark.read.csv('/Volumes/workspace/olist_ecom/olist_ecom_rawdata/olist_order_payments_dataset.csv',header=True,inferSchema=True)
reviews_df = spark.read.csv('/Volumes/workspace/olist_ecom/olist_ecom_rawdata/olist_order_reviews_dataset.csv',header=True,inferSchema=True)
prd_df = spark.read.csv('/Volumes/workspace/olist_ecom/olist_ecom_rawdata/olist_products_dataset.csv',header=True,inferSchema=True)
sellers_df = spark.read.csv('/Volumes/workspace/olist_ecom/olist_ecom_rawdata/olist_sellers_dataset.csv',header=True,inferSchema=True)
category_df = spark.read.csv('/Volumes/workspace/olist_ecom/olist_ecom_rawdata/product_category_name_translation.csv',header=True,inferSchema=True)

# Raw CSV → Delta Bronze Layer

cust_df.write.format("delta").mode("overwrite").saveAsTable("olist_bronze.customers")
orders_df.write.format("delta").mode("overwrite").saveAsTable("olist_bronze.orders")
geoloc_df.write.format("delta").mode("overwrite").saveAsTable("olist_bronze.geolocation")
orditems_df.write.format("delta").mode("overwrite").saveAsTable("olist_bronze.order_items")
payments_df.write.format("delta").mode("overwrite").saveAsTable("olist_bronze.payments")
reviews_df.write.format("delta").mode("overwrite").saveAsTable("olist_bronze.reviews")
prd_df.write.format("delta").mode("overwrite").saveAsTable("olist_bronze.products")
sellers_df.write.format("delta").mode("overwrite").saveAsTable("olist_bronze.sellers")
category_df.write.format("delta").mode("overwrite").saveAsTable("olist_bronze.category_translation")


display(orders_df.limit(5))

order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02T10:56:33.000Z,2017-10-02T11:07:15.000Z,2017-10-04T19:55:00.000Z,2017-10-10T21:25:13.000Z,2017-10-18T00:00:00.000Z
53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24T20:41:37.000Z,2018-07-26T03:24:27.000Z,2018-07-26T14:31:00.000Z,2018-08-07T15:27:45.000Z,2018-08-13T00:00:00.000Z
47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08T08:38:49.000Z,2018-08-08T08:55:23.000Z,2018-08-08T13:50:00.000Z,2018-08-17T18:06:29.000Z,2018-09-04T00:00:00.000Z
949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18T19:28:06.000Z,2017-11-18T19:45:59.000Z,2017-11-22T13:39:59.000Z,2017-12-02T00:28:42.000Z,2017-12-15T00:00:00.000Z
ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13T21:18:39.000Z,2018-02-13T22:20:29.000Z,2018-02-14T19:46:34.000Z,2018-02-16T18:17:02.000Z,2018-02-26T00:00:00.000Z


In [0]:
%sql SHOW TABLES IN olist_bronze;


database,tableName,isTemporary
olist_bronze,category_translation,false
olist_bronze,customers,false
olist_bronze,geolocation,false
olist_bronze,order_items,false
olist_bronze,orders,false
olist_bronze,payments,false
olist_bronze,products,false
olist_bronze,reviews,false
olist_bronze,sellers,false


In [0]:
%sql
SELECT COUNT(*) FROM olist_bronze.orders;


COUNT(*)
99441


In [0]:
%sql
-- Check for duplicate order IDs
SELECT COUNT(*) - COUNT(DISTINCT order_id) AS duplicate_orders
FROM olist_bronze.orders;


duplicate_orders
0
